In [ ]:
import pandas as pd

base = "../data/raw/_compare/"

a  = pd.read_csv(base + "customer_invoices/dataset.csv")
b1 = pd.read_csv(base + "global_b2b/train_invoices.csv", nrows=50000)
b2 = pd.read_csv(base + "global_b2b/test_invoices.csv",  nrows=50000)

for name, d in [("Customer Invoices", a), ("B2B train sample", b1), ("B2B test sample", b2)]:
    print("=====", name, d.shape)
    print(d.dtypes)
    print(d.isna().sum())
    print(d.head(3).T)

In [ ]:
print(list(b1.columns))
print(b1.isna().sum()[b1.isna().sum() > 0])
print(b1.head(3).T.to_string())


In [ ]:
# B2B: are there unpaid invoices?
print(b1["payment_status"].value_counts())
print(b1["is_overdue"].value_counts())
print((b1["payment_date"] < b1["invoice_date"]).sum())  # impossible dates (strings in ISO format compare correctly)

In [ ]:
print(b1.groupby("payment_status")["payment_date"].apply(lambda s: s.isna().sum()))
print(b1[b1["payment_status"] != "Paid"][["invoice_date","due_date","payment_date","payment_status"]].head(10))

In [ ]:
unpaid = b1[b1["payment_status"] != "Paid"]
paid   = b1[b1["payment_status"] == "Paid"]

print("unpaid, payment_date == due_date:", (unpaid["payment_date"] == unpaid["due_date"]).mean())
print("paid, payment_date == due_date:  ", (paid["payment_date"] == paid["due_date"]).mean())

In [ ]:
print(pd.crosstab(a["isOpen"], a["clear_date"].isna()))
print("duplicate invoice_id:", a["invoice_id"].duplicated().sum())
print("duplicate doc_id:", a["doc_id"].duplicated().sum())

In [ ]:
print(a["invoice_id"].isna().sum(), a["doc_id"].isna().sum())
print("fully identical rows:", a.duplicated().sum())
dups = a[a["invoice_id"].duplicated(keep=False) & a["invoice_id"].notna()].sort_values("invoice_id")
print(dups.head(10).T.to_string())

In [ ]:
# 1. Is invoice_id always the same as doc_id?
both = a.dropna(subset=["invoice_id"])
print("invoice_id == doc_id:", (both["invoice_id"] == both["doc_id"]).mean())

# 2. What are the 6 rows with missing invoice_id?
print(a[a["invoice_id"].isna()].T.to_string())

# 3. After dropping exact copies, is doc_id unique?
a2 = a.drop_duplicates()
print(a2.shape, "duplicate doc_id left:", a2["doc_id"].duplicated().sum())


In [ ]:
a2 = a.drop_duplicates().copy()
a2["invoice_id"] = a2["invoice_id"].fillna(a2["doc_id"])
print("missing invoice_id:", a2["invoice_id"].isna().sum(), "| unique:", a2["invoice_id"].is_unique)

def ymd(s):
    return pd.to_datetime(s.astype("Int64").astype(str), format="%Y%m%d", errors="coerce")

d = pd.DataFrame({
    "posting":  pd.to_datetime(a2["posting_date"], errors="coerce"),
    "baseline": ymd(a2["baseline_create_date"]),
    "due":      ymd(a2["due_in_date"]),
    "clear":    pd.to_datetime(a2["clear_date"], errors="coerce"),
})
print(d.isna().sum())
print("due < posting:  ", (d.due < d.posting).sum())
print("due < baseline: ", (d.due < d.baseline).sum())
print("clear < baseline:", (d.clear < d.baseline).sum())
print("clear < posting: ", (d.clear < d.posting).sum())
print("baseline == document_create_date.1:",
      (a2["baseline_create_date"] == a2["document_create_date.1"]).mean())

In [ ]:

datasets = {
    "Customer Invoices": a,
    "Global B2B Train": b1,
    "Global B2B Test": b2
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nSample records:")
    display(df.head(5))


In [ ]:

# 1. Check invoice status distribution
print("Invoice Status Distribution:")
print(a["isOpen"].value_counts(dropna=False))

# 2. Check clear_date missing values by invoice status
print("\nClear Date Missing vs Invoice Status:")
print(pd.crosstab(a["isOpen"], a["clear_date"].isna()))

# 3. Check duplicate records
print("\nExact duplicate records:", a.duplicated().sum())

# 4. Check duplicate invoice IDs
print("Duplicate invoice IDs:", a["invoice_id"].duplicated().sum())

# 5. Inspect duplicate invoice records
duplicates = a[a.duplicated(subset=["invoice_id"], keep=False)]
display(duplicates.head(10))

# 6. Check completely empty columns
print("\nCompletely empty columns:")
print(a.columns[a.isna().all()].tolist())

# 7. Check date column samples
date_columns = [
    "clear_date",
    "posting_date",
    "document_create_date",
    "due_in_date",
    "baseline_create_date"
]

for col in date_columns:
    print(f"\n{col}")
    print(a[col].head(5).tolist())



In [ ]:

# Find records with repeated invoice IDs
duplicate_ids = a[
    a.duplicated(subset=["invoice_id"], keep=False)
].sort_values("invoice_id")

print("Total records with repeated invoice IDs:",
      len(duplicate_ids))

print("\nNumber of unique repeated invoice IDs:",
      duplicate_ids["invoice_id"].nunique())

# Find invoice IDs with conflicting records
duplicate_summary = (
    duplicate_ids.groupby("invoice_id")
    .agg(
        record_count=("invoice_id", "size"),
        customer_count=("cust_number", "nunique"),
        amount_count=("total_open_amount", "nunique"),
        due_date_count=("due_in_date", "nunique"),
        clear_date_count=("clear_date", "nunique")
    )
    .reset_index()
)

print("\nRepeated invoice ID summary:")
display(duplicate_summary.head(20))

print("\nInvoice IDs with conflicting amounts:")
display(
    duplicate_summary[
        duplicate_summary["amount_count"] > 1
    ].head(10)
)


In [ ]:

# Identify duplicate invoice IDs
dup = a[a.duplicated("invoice_id", keep=False)].copy()

# Find columns that differ within each repeated invoice ID
varying_columns = []

for col in a.columns:
    counts = dup.groupby("invoice_id")[col].nunique(dropna=False)
    if (counts > 1).any():
        varying_columns.append(col)

print("Columns with conflicting values:", varying_columns)

# Display records where at least one column differs
if varying_columns:
    conflict_ids = (
        dup.groupby("invoice_id")[varying_columns]
        .nunique(dropna=False)
        .gt(1)
        .any(axis=1)
    )

    conflict_invoice_ids = conflict_ids[conflict_ids].index

    print("Invoice IDs with conflicting records:",
          len(conflict_invoice_ids))

    display(
        dup[dup["invoice_id"].isin(conflict_invoice_ids)]
        .sort_values("invoice_id")
        .to_string(index=False)
    )
else:
    print("No conflicting values found.")


In [ ]:

print("Original rows:", len(a))
print("Exact duplicate rows:", a.duplicated().sum())
print("Rows after exact duplicate removal:",
      len(a.drop_duplicates()))

print("Unique invoice IDs:", a["invoice_id"].nunique())
print("Repeated invoice IDs:",
      a["invoice_id"].duplicated().sum())
      

print("Unique customers:", a["cust_number"].nunique())


In [ ]:

# Work on a copy after exact duplicate removal
a_dedup = a.drop_duplicates().copy()

print("Missing invoice IDs:", a_dedup["invoice_id"].isna().sum())

print("\nRepeated invoice IDs after exact deduplication:")
print(
    a_dedup[
        a_dedup.duplicated("invoice_id", keep=False)
    ].sort_values("invoice_id").to_string(index=False)
)

print("\nUnique non-null invoice IDs:",
      a_dedup["invoice_id"].nunique())

print("Rows after exact deduplication:", len(a_dedup))


In [ ]:

date_columns = [
    "clear_date",
    "posting_date",
    "document_create_date",
    "due_in_date",
    "baseline_create_date"
]

# Convert dates in the working copy
for col in date_columns:
    a_dedup[col] = pd.to_datetime(
        a_dedup[col], errors="coerce"
    )

print("DATE VALIDATION")

for col in date_columns:
    print(
        col,
        "| Missing:", a_dedup[col].isna().sum(),
        "| Min:", a_dedup[col].min(),
        "| Max:", a_dedup[col].max()
    )

print("\nInvalid date relationships")

print("Posting before document creation:",
      (a_dedup["posting_date"] <
       a_dedup["document_create_date"]).sum())

print("Due date before posting:",
      (a_dedup["due_in_date"] <
       a_dedup["posting_date"]).sum())

print("Payment before posting:",
      (a_dedup["clear_date"] <
       a_dedup["posting_date"]).sum())


In [ ]:

# Create a fresh working copy
cleaning_df = a.drop_duplicates().copy()

# Date columns stored as YYYYMMDD numbers
numeric_date_columns = [
    "document_create_date",
    "document_create_date.1",
    "due_in_date",
    "baseline_create_date"
]

# Convert numeric YYYYMMDD values correctly
for col in numeric_date_columns:
    values = pd.to_numeric(cleaning_df[col], errors="coerce")

    date_strings = values.map(
        lambda x: str(int(x)) if pd.notna(x) else None
    )

    cleaning_df[col] = pd.to_datetime(
        date_strings,
        format="%Y%m%d",
        errors="coerce"
    )

# Convert standard date columns
for col in ["clear_date", "posting_date"]:
    cleaning_df[col] = pd.to_datetime(
        cleaning_df[col],
        errors="coerce"
    )

print("CORRECTED DATE VALIDATION")

for col in [
    "clear_date",
    "posting_date",
    "document_create_date",
    "document_create_date.1",
    "due_in_date",
    "baseline_create_date"
]:
    print(
        f"{col}:",
        "Missing =", cleaning_df[col].isna().sum(),
        "| Min =", cleaning_df[col].min(),
        "| Max =", cleaning_df[col].max()
    )


In [ ]:

print("DATE RELATIONSHIP VALIDATION")

checks = {
    "Posting before document creation":
        cleaning_df["posting_date"] <
        cleaning_df["document_create_date"],

    "Due date before posting":
        cleaning_df["due_in_date"] <
        cleaning_df["posting_date"],

    "Payment before posting":
        cleaning_df["clear_date"] <
        cleaning_df["posting_date"],

    "Payment before due date":
        cleaning_df["clear_date"] <
        cleaning_df["due_in_date"]
}

for name, condition in checks.items():
    print(f"{name}: {condition.sum()}")


In [ ]:
date_issue_summary = pd.DataFrame({
    "Issue": list(checks.keys()),
    "Affected Records": [
        condition.sum() for condition in checks.values()
    ]
})

display(date_issue_summary)

In [ ]:
print("IDENTIFIER VALIDATION")

print("Missing invoice IDs:", cleaning_df["invoice_id"].isna().sum())
print("Unique invoice IDs:", cleaning_df["invoice_id"].nunique())
print("Missing document IDs:", cleaning_df["doc_id"].isna().sum())
print("Unique document IDs:", cleaning_df["doc_id"].nunique())

print(
    "Duplicate document IDs:",
    cleaning_df["doc_id"].duplicated().sum()
)

In [ ]:
quality_report = pd.DataFrame({
    "Column": cleaning_df.columns,
    "Data Type": cleaning_df.dtypes.astype(str).values,
    "Missing Values": cleaning_df.isna().sum().values,
    "Missing Percentage": (
        cleaning_df.isna().mean().values * 100
    ).round(2)
})

display(quality_report)

In [ ]:
quality_report.to_csv(
    "dataset_quality_report.csv",
    index=False
)

date_issue_summary.to_csv(
    "date_validation_report.csv",
    index=False
)

print("Investigation reports saved.")

In [ ]:
# Remove completely empty columns
empty_columns = [
    col for col in cleaning_df.columns
    if cleaning_df[col].isna().all()
]

cleaning_df = cleaning_df.drop(columns=empty_columns)

print("Removed columns:", empty_columns)
print("Updated shape:", cleaning_df.shape)

In [ ]:
cleaning_log = pd.DataFrame([
    ["Duplicate removal", "Removed 1,161 exact duplicate rows"],
    ["Empty columns", "Removed area_business (100% missing)"],
    ["Missing clear_date", "Preserved; represents open invoices"],
    ["Missing invoice_id", "Preserved; doc_id is unique"],
    ["Date inconsistencies", "Flagged; records not automatically deleted"],
    ["Early payments", "Preserved; potentially valid payments"]
], columns=["Operation", "Decision"])

display(cleaning_log)

cleaning_log.to_csv(
    "cleaning_log.csv",
    index=False
)

In [ ]:
cleaning_df.to_csv(
    "customer_invoices_cleaned.csv",
    index=False
)

print("Cleaned dataset exported successfully!")
print("Final shape:", cleaning_df.shape)

In [ ]:
print("GLOBAL B2B DATASET COMPARISON")
print("-" * 45)

print("Training dataset shape:", b1.shape)
print("Testing dataset shape:", b2.shape)

print("\nTraining columns:")
print(b1.columns.tolist())

print("\nTesting columns:")
print(b2.columns.tolist())

print("\nTraining missing values:")
display(b1.isnull().sum())

print("\nTesting missing values:")
display(b2.isnull().sum())

print("\nTraining sample:")
display(b1.head())

print("\nTesting sample:")
display(b2.head())

In [ ]:
for name, data in [("TRAIN", b1), ("TEST", b2)]:

    print(f"\n{'='*15} {name} DATASET {'='*15}")

    print("Shape:", data.shape)
    print("Duplicate rows:", data.duplicated().sum())
    print("Duplicate invoice IDs:", data["invoice_id"].duplicated().sum())

    print("\nMissing values:")
    print(data.isnull().sum()[lambda x: x > 0])

    print("\nPayment status distribution:")
    print(data["payment_status"].value_counts(dropna=False))

    print("\nOverdue distribution:")
    print(data["is_overdue"].value_counts(dropna=False))

    print("\nDate ranges:")
    for col in ["invoice_date", "due_date", "payment_date"]:
        dates = pd.to_datetime(data[col], errors="coerce")
        print(
            col,
            "| Min:", dates.min(),
            "| Max:", dates.max(),
            "| Invalid:", dates.isna().sum()
        )

In [ ]:
print("PAYMENT STATUS vs OVERDUE")

display(pd.crosstab(
    b1["payment_status"],
    b1["is_overdue"],
    margins=True
))

print("\nPAYMENT DATE AVAILABILITY")

display(
    b1.groupby("payment_status")["payment_date"]
    .agg(
        total="count",
        missing=lambda x: x.isna().sum()
    )
)

print("\nDATE CONSISTENCY")

for name, data in [("TRAIN", b1), ("TEST", b2)]:
    temp = data.copy()

    for col in ["invoice_date", "due_date", "payment_date"]:
        temp[col] = pd.to_datetime(temp[col], errors="coerce")

    print(f"\n{name}")
    print("Due date before invoice date:",
          (temp["due_date"] < temp["invoice_date"]).sum())
    print("Payment before invoice date:",
          (temp["payment_date"] < temp["invoice_date"]).sum())
    print("Payment after due date:",
          (temp["payment_date"] > temp["due_date"]).sum())

In [ ]:
# Investigate payment dates and overdue labels

temp = b1.copy()

for col in ["invoice_date", "due_date", "payment_date"]:
    temp[col] = pd.to_datetime(temp[col], errors="coerce")

temp["calculated_days_to_pay"] = (
    temp["payment_date"] - temp["invoice_date"]
).dt.days

temp["calculated_delay"] = (
    temp["payment_date"] - temp["due_date"]
).dt.days

print("PAYMENT STATUS SUMMARY")
print(temp.groupby("payment_status").agg(
    invoices=("invoice_id", "count"),
    avg_days_to_pay=("calculated_days_to_pay", "mean"),
    min_delay=("calculated_delay", "min"),
    max_delay=("calculated_delay", "max")
).round(2))

print("\nOVERDUE LABEL CHECK")
print(temp.loc[
    temp["payment_status"] == "Overdue",
    ["invoice_id", "invoice_date", "due_date",
     "payment_date", "payment_status", "is_overdue"]
].head(5).to_string(index=False))

In [ ]:
# Validate payment-delay targets in Customer Invoices

temp = cleaning_df.copy()

temp["days_to_pay"] = (
    temp["clear_date"] - temp["posting_date"]
).dt.days

temp["delay"] = (
    temp["clear_date"] - temp["due_in_date"]
).dt.days

print("PAYMENT STATUS")
print(temp["isOpen"].value_counts(dropna=False))

print("\nDAYS TO PAY")
print(temp.groupby("isOpen")["days_to_pay"].agg(
    ["count", "mean", "min", "max"]
).round(2))

print("\nPAYMENT DELAY")
print(temp.groupby("isOpen")["delay"].agg(
    ["count", "mean", "min", "max"]
).round(2))

print("\nPAYMENT DATE CONSISTENCY")
print("Paid invoices with missing clear_date:",
      temp.loc[temp["isOpen"] == 0, "clear_date"].isna().sum())

print("Open invoices with clear_date:",
      temp.loc[temp["isOpen"] == 1, "clear_date"].notna().sum())

In [ ]:
# Create payment prediction targets

cleaning_df["days_to_pay"] = (
    cleaning_df["clear_date"] - cleaning_df["posting_date"]
).dt.days

cleaning_df["delay"] = (
    cleaning_df["clear_date"] - cleaning_df["due_in_date"]
).dt.days

cleaning_df["paid_late"] = (
    cleaning_df["delay"] > 0
).where(cleaning_df["isOpen"] == 0, pd.NA)

cleaning_df["open_paid_flag"] = (
    cleaning_df["isOpen"].map({0: 1, 1: 0})
)

print("Final shape:", cleaning_df.shape)
print(cleaning_df[
    ["days_to_pay", "delay", "paid_late", "open_paid_flag"]
].head())

In [ ]:
cleaning_df["paid_late"] = cleaning_df["paid_late"].astype("Int64")

cleaning_df.to_csv("customer_invoices_cleaned.csv", index=False)

print("Dataset exported successfully!")
print("Shape:", cleaning_df.shape)
print("\nTarget distribution:")
print(cleaning_df["paid_late"].value_counts(dropna=False))

In [ ]:
data_dictionary = pd.DataFrame({
    "Column": cleaning_df.columns,
    "Data Type": cleaning_df.dtypes.astype(str).values,
    "Description": [
        "See original dataset column definition"
        for _ in cleaning_df.columns
    ]
})

target_descriptions = {
    "days_to_pay": "Days between posting date and payment date",
    "delay": "Days between due date and payment date; positive means late",
    "paid_late": "1 = paid late, 0 = paid on time or early, NA = open",
    "open_paid_flag": "1 = paid invoice, 0 = open invoice"
}

data_dictionary["Description"] = [
    target_descriptions.get(col, desc)
    for col, desc in zip(
        data_dictionary["Column"],
        data_dictionary["Description"]
    )
]

data_dictionary.to_csv("data_dictionary.csv", index=False)

print("Data dictionary exported:", data_dictionary.shape)